# Imputation

### Prepare the data

In [ ]:
import pandas as pd

feature_matrix = pd.read_parquet('../data/intermediate/feature_matrix.parquet')
feature_matrix.drop(columns=["spliceai", "gene", "enst"], inplace=True, errors="ignore")
feature_matrix

,ID,ensg,SIFT_score,Polyphen2_HDIV_score,Polyphen2_HVAR_score,MutationTaster_score,MutationAssessor_score,PROVEAN_score,VEST4_score,MetaSVM_score,...,pDN,DN_pctl,pGOF,GOF_pctl,pLOF,LOF_pctl,esm2_mean_log_likelihood_difference,esm2_perplexity_difference,esm2_cosine_similarity,esm2_avg_abs_difference
0,10-100057090-C-T,ENSG00000120054,0.180,0.002,0.003,0.24,0.820,-1.72,0.166,-0.9950,...,0.692004,0.370096,0.627789,0.458777,0.284495,0.711122,-0.001133,0.001560,0.999975,0.001065
1,10-100069757-C-T,ENSG00000120054,0.171,0.863,NaN,0.24,2.275,-4.19,NaN,NaN,...,0.692004,0.370096,0.627789,0.458777,0.284495,0.711122,0.000762,-0.001011,0.999961,0.001273
2,10-100076062-G-A,ENSG00000120054,1.000,0.005,0.002,0.10,-1.620,1.49,0.110,-0.9202,...,0.692004,0.370096,0.627789,0.458777,0.284495,0.711122,0.006985,-0.008888,0.999914,0.001779
3,10-100081405-G-A,ENSG00000120054,0.662,0.001,0.008,0.05,0.330,-1.43,0.104,-0.9218,...,0.692004,0.370096,0.627789,0.458777,0.284495,0.711122,-0.009512,0.012117,0.999850,0.002473
4,10-100152307-T-C,ENSG00000107566,0.138,NaN,NaN,0.02,NaN,-0.79,NaN,-0.9499,...,0.766297,0.161208,0.631910,0.448466,0.279330,0.725755,-0.002086,0.002513,0.999942,0.001543
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1117936,X-154442668-A-C,ENSG00000203879,0.000,0.930,0.276,0.31,0.000,-0.94,0.437,-0.6034,...,0.238068,0.988264,0.424269,0.822539,0.709899,0.080678,-0.001493,0.002581,0.999020,0.008181
1117937,X-154442668-A-G,ENSG00000203879,0.000,0.651,0.058,0.27,0.000,-0.69,0.541,-0.6869,...,0.238068,0.988264,0.424269,0.822539,0.709899,0.080678,-0.001625,0.002809,0.999335,0.007607
1117938,X-154442668-A-T,ENSG00000203879,0.000,0.651,0.107,0.26,0.000,-1.05,0.555,-0.6594,...,0.238068,0.988264,0.424269,0.822539,0.709899,0.080678,0.003814,-0.006574,0.999617,0.005875
1117939,X-154442669-G-C,ENSG00000203879,0.000,0.006,0.003,0.27,0.000,-0.87,0.504,-0.6444,...,0.238068,0.988264,0.424269,0.822539,0.709899,0.080678,-0.001080,0.001866,0.999437,0.006611


### Train imputation models (LightGBM regressors) for features with informative or high missingness and impute missing inputs

- We exclude tools with unknown training sets from the imputation models, because we could not identify and remove their scores for their training variants and we want to avoid information leakage.
- To avoid cascading imputation, we always use a frozen copy of the original (pre-imputation) feature matrix as predictors, so imputed values are never used as inputs for subsequent imputations.

In [ ]:
import os
import joblib
import pandas as pd
import lightgbm as lgb

with open("../resources/feature_lists/columns_to_impute.txt", "r") as f:
    columns_to_impute = [line.strip() for line in f.readlines()]

with open("../resources/feature_lists/veps_excluded_due_to_unavailable_training_sets.txt", "r") as f:
    no_training_set = [line.strip() for line in f]

imputed_file = "../data/intermediate/feature_matrix_imputed.parquet"
model_dir = "../models/imputation"
os.makedirs(model_dir, exist_ok=True)

df = feature_matrix.copy()

non_predictors = {"ID", "ensg"}
numeric_cols = [col for col in df.columns if col not in non_predictors]
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors="coerce")

imputed_values = {}

for col in columns_to_impute:
    if col not in df.columns:
        continue
    if col in no_training_set:
        continue

    print(f"Imputing {col}...")

    df_non_null = df.dropna(subset=[col])
    df_null = df[df[col].isnull()]

    if df_non_null.empty or df_null.empty:
        print(f"  Skipped {col} (no train/predict data).")
        continue

    predictors = [f for f in df.columns if f not in non_predictors and f != col and f not in no_training_set]

    X_train = df_non_null[predictors].copy()
    y_train = df_non_null[col]
    X_pred = df_null[predictors].copy()

    X_train = X_train.dropna(axis=1, how="all")
    X_pred = X_pred[X_train.columns]

    model = lgb.LGBMRegressor(
        objective="regression",
        random_state=42,
        n_estimators=250,
        learning_rate=0.06,
        num_leaves=128,
        max_depth=-1,
        min_child_samples=500,
        reg_lambda=1.0,
        reg_alpha=0.0,
        subsample=0.7,
        subsample_freq=1,
        colsample_bytree=0.3,
        n_jobs=8,
        verbose=-1,
    )

    model.fit(X_train, y_train)

    model_path = os.path.join(model_dir, f"{col}_imputer.pkl")
    joblib.dump(model, model_path)

    imputed_values[col] = model.predict(X_pred)

for col, preds in imputed_values.items():
    missing_mask = df[col].isnull()
    df.loc[missing_mask, col] = preds

df.to_parquet(imputed_file, index=False)
print(f"\nImputed feature_matrix saved to: {imputed_file}")


Imputing glm_CaddDeogenRevel...
Imputing glm_AlphDeogenRevel...
Imputing glm_AlphCaddDeogen...
Imputing glm_AlphRevelCadd...
Imputing glm_AlphRevel...
Imputing MutPred_score...
Imputing glm_DeogenRevel...
Imputing glm_RevelCadd...
Imputing REVEL_score...
Imputing MetaRNN_score...
Imputing M_CAP_score...
Imputing EVH_epistatic...
Imputing EVH_independent...
Imputing VARITY_ER_LOO_score...
Imputing VARITY_R_LOO_score...
Imputing VARITY_ER_score...
Imputing VARITY_R_score...
Imputing MutFormer_score...
Imputing MetaLR_score...
Imputing MetaSVM_score...
Imputing glm_AlphDeogen...
Imputing VEST4_score...
Imputing fathmm_XF_coding_score...
Imputing MutScore_score...
Imputing glm_CaddDeogen...
Imputing DEOGEN2_score...
Imputing ClinPred_score...
Imputing MPC_score...
Imputing sigma_score...
Imputing glm_AlphCadd...
Imputing EWSIM...
Imputing ESM1v...
Imputing GERP_91_mammals...
Imputing popEVE...
Imputing Polyphen2_HVAR_score...
Imputing PHACT...
Imputing LIST_S2_score...
Imputing MTR...
Impu